# Cane / Walker Single-IMU Training

This notebook trains a first-pass movement classifier that uses exactly one 6-axis IMU stream: `acc_x`, `acc_y`, `acc_z`, `gyro_x`, `gyro_y`, and `gyro_z`.

The HuGaDB data in this repo has multiple body-worn IMUs, so the notebook deliberately selects only one sensor pair as a proxy. That keeps the training shape compatible with a real Modulino Movement mounted on a cane or walker. The proxy is useful for building the pipeline, feature extraction, and evaluation flow, but final model quality should be validated with real cane/walker recordings.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
HUGADB_DIR = PROJECT_ROOT / "data" / "hugadb"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# HuGaDB sampling is about 60 Hz. Set this to your firmware sampling rate
# when training on real cane/walker CSVs.
SAMPLE_RATE_HZ = 60
WINDOW_SECONDS = 2.0
STEP_SECONDS = 1.0

# One HuGaDB body sensor is used as a stand-in for a single cane/walker IMU.
# Options in the dataset include: right_foot, right_shin, right_thigh,
# left_foot, left_shin, left_thigh.
HUGADB_PROXY_SENSOR = "right_foot"

# Keep the first notebook run quick. Change to None to train on every HuGaDB file.
HUGADB_MAX_FILES = 8
MAX_WINDOWS = 3000

ACC_COLS = ["acc_x", "acc_y", "acc_z"]
GYRO_COLS = ["gyro_x", "gyro_y", "gyro_z"]
IMU_COLS = ACC_COLS + GYRO_COLS

ACTIVITY_GROUPS = {
    "walking": "walk",
    "running": "walk",
    "going up": "stairs",
    "going down": "stairs",
    "standing": "stationary",
    "sitting": "stationary",
    "sitting down": "transition",
    "standing up": "transition",
}


In [ ]:
def normalize_hugadb_sensor(df: pd.DataFrame, sensor_name: str) -> pd.DataFrame:
    """Keep one HuGaDB IMU and rename it to the cane/walker schema."""
    source_cols = {
        "acc_x": f"accelerometer_{sensor_name}_x",
        "acc_y": f"accelerometer_{sensor_name}_y",
        "acc_z": f"accelerometer_{sensor_name}_z",
        "gyro_x": f"gyroscope_{sensor_name}_x",
        "gyro_y": f"gyroscope_{sensor_name}_y",
        "gyro_z": f"gyroscope_{sensor_name}_z",
    }
    missing = [col for col in source_cols.values() if col not in df.columns]
    if missing:
        raise ValueError(f"Missing HuGaDB columns for {sensor_name}: {missing}")

    out = df[list(source_cols.values()) + ["activity"]].rename(
        columns={source: target for target, source in source_cols.items()}
    )
    out["activity"] = out["activity"].astype(str).str.strip().str.lower().str.replace("_", " ", regex=False)
    out["label"] = out["activity"].map(ACTIVITY_GROUPS)
    return out.dropna(subset=["label"]).reset_index(drop=True)


def load_hugadb_single_sensor(data_dir: Path, sensor_name: str, max_files: int | None = None) -> pd.DataFrame:
    frames = []
    csv_paths = sorted(data_dir.glob("HuGaDB_v2_various_*.csv"))
    if max_files is not None:
        csv_paths = csv_paths[:max_files]

    for path in csv_paths:
        raw = pd.read_csv(path, index_col=0)
        frame = normalize_hugadb_sensor(raw, sensor_name)
        frame["source_file"] = path.name
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(f"No HuGaDB CSV files found in {data_dir}")

    return pd.concat(frames, ignore_index=True)


data = load_hugadb_single_sensor(HUGADB_DIR, HUGADB_PROXY_SENSOR, max_files=HUGADB_MAX_FILES)
data.head()


In [ ]:
def _axis_features(values: np.ndarray, prefix: str) -> dict[str, float]:
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_std": float(np.std(values)),
        f"{prefix}_min": float(np.min(values)),
        f"{prefix}_max": float(np.max(values)),
        f"{prefix}_range": float(np.max(values) - np.min(values)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_iqr": float(np.percentile(values, 75) - np.percentile(values, 25)),
        f"{prefix}_rms": float(np.sqrt(np.mean(np.square(values)))),
        f"{prefix}_energy": float(np.mean(np.square(values))),
    }


def _peak_count(values: np.ndarray) -> int:
    if len(values) < 3:
        return 0
    centered = values - np.mean(values)
    threshold = np.std(centered) * 0.5
    peaks = (centered[1:-1] > centered[:-2]) & (centered[1:-1] > centered[2:]) & (centered[1:-1] > threshold)
    return int(np.sum(peaks))


def window_features(window: pd.DataFrame) -> dict[str, float]:
    features = {}

    for col in IMU_COLS:
        features.update(_axis_features(window[col].to_numpy(dtype=float), col))

    acc_mag = np.linalg.norm(window[ACC_COLS].to_numpy(dtype=float), axis=1)
    gyro_mag = np.linalg.norm(window[GYRO_COLS].to_numpy(dtype=float), axis=1)
    features.update(_axis_features(acc_mag, "acc_mag"))
    features.update(_axis_features(gyro_mag, "gyro_mag"))
    features["acc_mag_peak_count"] = _peak_count(acc_mag)
    features["gyro_mag_peak_count"] = _peak_count(gyro_mag)
    return features


def make_windows(df: pd.DataFrame, sample_rate_hz: int, window_seconds: float, step_seconds: float) -> pd.DataFrame:
    window_size = int(round(sample_rate_hz * window_seconds))
    step_size = int(round(sample_rate_hz * step_seconds))
    rows = []

    for source_file, file_df in df.groupby("source_file", sort=False):
        file_df = file_df.reset_index(drop=True)
        for start in range(0, len(file_df) - window_size + 1, step_size):
            window = file_df.iloc[start : start + window_size]
            label_counts = window["label"].value_counts(normalize=True)
            label = label_counts.index[0]

            # Skip mixed windows around transitions unless one class dominates.
            if label_counts.iloc[0] < 0.70:
                continue

            features = window_features(window)
            features["label"] = label
            features["source_file"] = source_file
            features["start_sample"] = start
            rows.append(features)

    return pd.DataFrame(rows)


windows = make_windows(data, SAMPLE_RATE_HZ, WINDOW_SECONDS, STEP_SECONDS)
print(windows["label"].value_counts())
windows.head()


In [ ]:
feature_cols = [col for col in windows.columns if col not in {"label", "source_file", "start_sample"}]
X = windows[feature_cols]
y = windows["label"]
groups = windows["source_file"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print(classification_report(y_test, pred))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, xticks_rotation=45, normalize="true")
ax.set_title("Single-IMU proxy classifier")
plt.tight_layout()
plt.show()

importances = model.named_steps["classifier"].feature_importances_
importance = (
    pd.DataFrame({"feature": feature_cols, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance["feature"][::-1], importance["importance"][::-1])
ax.set_title("Top feature importances")
ax.set_xlabel("Random forest importance")
plt.tight_layout()
plt.show()


In [ ]:
artifact = {
    "model": model,
    "feature_cols": feature_cols,
    "imu_cols": IMU_COLS,
    "sample_rate_hz": SAMPLE_RATE_HZ,
    "window_seconds": WINDOW_SECONDS,
    "step_seconds": STEP_SECONDS,
    "proxy_sensor": HUGADB_PROXY_SENSOR,
    "labels": sorted(y.unique().tolist()),
}

model_path = MODEL_DIR / "cane_imu_random_forest.joblib"
joblib.dump(artifact, model_path)
model_path


In [ ]:
def prepare_cane_dataframe(
    cane_df: pd.DataFrame,
    column_map: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Return a dataframe with acc_x/y/z and gyro_x/y/z columns.

    For real Modulino Movement data, pass a map like:
    {
        "acc_x": "accelerometer_x",
        "acc_y": "accelerometer_y",
        "acc_z": "accelerometer_z",
        "gyro_x": "gyroscope_x",
        "gyro_y": "gyroscope_y",
        "gyro_z": "gyroscope_z",
    }
    """
    column_map = column_map or {col: col for col in IMU_COLS}
    missing = [source for source in column_map.values() if source not in cane_df.columns]
    if missing:
        raise ValueError(f"Missing cane IMU columns: {missing}")

    return cane_df[[column_map[col] for col in IMU_COLS]].rename(
        columns={source: target for target, source in column_map.items()}
    )


def predict_cane_windows(cane_df: pd.DataFrame, artifact: dict) -> pd.DataFrame:
    normalized = prepare_cane_dataframe(cane_df)
    window_size = int(round(artifact["sample_rate_hz"] * artifact["window_seconds"]))
    step_size = int(round(artifact["sample_rate_hz"] * artifact["step_seconds"]))
    rows = []

    for start in range(0, len(normalized) - window_size + 1, step_size):
        window = normalized.iloc[start : start + window_size]
        features = window_features(window)
        rows.append({"start_sample": start, **features})

    if not rows:
        return pd.DataFrame(columns=["start_sample", "prediction", "confidence"])

    feature_frame = pd.DataFrame(rows)
    probabilities = artifact["model"].predict_proba(feature_frame[artifact["feature_cols"]])
    class_names = artifact["model"].classes_
    best_idx = np.argmax(probabilities, axis=1)

    result = feature_frame[["start_sample"]].copy()
    result["prediction"] = class_names[best_idx]
    result["confidence"] = probabilities[np.arange(len(best_idx)), best_idx]
    return result


# Smoke test inference on the first few seconds of the proxy data.
loaded_artifact = joblib.load(model_path)
predict_cane_windows(data[IMU_COLS].head(SAMPLE_RATE_HZ * 10), loaded_artifact).head()
